#Qdrant Hybrid Search (No Agents, No LLM)


This notebook demonstrates Qdrant's hybrid search capabilities:
- Dense vectors (semantic similarity via BGE-small)
- Sparse vectors (keyword matching via BM42)
- RRF fusion (Reciprocal Rank Fusion)
No agents, no llm, just pure vector search.

Prerequisite:
Run data ingestion first in notebook 1



In [1]:
#pip install qdrant-client[fastembed]

In [ ]:
import os
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding

from google.colab import userdata

# Configuration
QDRANT_URL = userdata.get('QDRANT_CLOUD_URL')
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
COLLECTION_NAME = "california_housing_statutes_full"

print("✓ Configuration loaded")

✓ Configuration loaded


## Step 1: Initialize Qdrant Client


In [ ]:
print("\n" + "="*80)
print("QDRANT HYBRID SEARCH DEMO")
print("Dense + Sparse vectors with RRF fusion")
print("="*80)

print("\nStep 1: Connecting to Qdrant...")
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print("✓ Connected to Qdrant Cloud\n")



QDRANT HYBRID SEARCH DEMO
Dense + Sparse vectors with RRF fusion

Step 1: Connecting to Qdrant...
✓ Connected to Qdrant Cloud



## Step 2: Load Embedding Models


In [ ]:
print("Step 2: Loading embedding models...")
dense_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm42-all-minilm-l6-v2-attentions")
print("✓ Models loaded (Dense: BGE-small, Sparse: BM42)\n")


Step 2: Loading embedding models...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

stopwords.txt:   0%|          | 0.00/936 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

✓ Models loaded (Dense: BGE-small, Sparse: BM42)



## Step 3: Define Search Query


In [ ]:
# Change this to search for different topics
query = "Can a landlord evict a tenant without notice?"
print(f"Query: {query}\n")


Query: Can a landlord evict a tenant without notice?



## Step 4: Generate Query Embeddings


In [ ]:
print("Step 3: Generating query embeddings...")
dense_query = list(dense_model.embed([query]))[0].tolist()
sparse_query_embedding = list(sparse_model.embed([query]))[0]
sparse_query = models.SparseVector(
    indices=sparse_query_embedding.indices.tolist(),
    values=sparse_query_embedding.values.tolist()
)
print(f"  → Dense vector: {len(dense_query)} dimensions")
print(f"  → Sparse vector: {len(sparse_query.indices)} tokens")
print("✓ Query embeddings generated\n")


Step 3: Generating query embeddings...
  → Dense vector: 384 dimensions
  → Sparse vector: 5 tokens
✓ Query embeddings generated



## Step 5: Execute Hybrid Search with RRF


In [ ]:
print("Step 4: Executing Qdrant hybrid search...")
print("  → Prefetch: Dense search (top 20)")
print("  → Prefetch: Sparse search (top 20)")
print("  → Fusion: RRF (Reciprocal Rank Fusion)\n")

results = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        models.Prefetch(query=dense_query, using="dense", limit=20),
        models.Prefetch(query=sparse_query, using="sparse", limit=20),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
)

print(f"✓ Retrieved {len(results.points)} statutes\n")


Step 4: Executing Qdrant hybrid search...
  → Prefetch: Dense search (top 20)
  → Prefetch: Sparse search (top 20)
  → Fusion: RRF (Reciprocal Rank Fusion)

✓ Retrieved 5 statutes



## Step 6: Display Search Results


In [ ]:
print("="*80)
print("SEARCH RESULTS")
print("="*80 + "\n")

for i, point in enumerate(results.points, 1):
    citation = point.payload.get('citation', 'Unknown')
    text = point.payload.get('text', '')
    score = point.score

    print(f"[{i}] {citation}")
    print(f"    RRF Score: {score:.3f}")
    print(f"    {text[:200]}...")
    print()


SEARCH RESULTS

[1] CA Health & Safety Code § 34331 (2021)
    RRF Score: 0.667
    34331.  In the operation or management of housing projects an authority shall not do any of the following:(a)  Evict any tenant without reasonable cause and unless the tenant has been given a written ...

[2] CA Civ Pro Code § 1179.04 (2021)
    RRF Score: 0.500
    1179.04.  (a) On or before September 30, 2020, a landlord shall provide, in at least 12-point type, the following notice to tenants who, as of September 1, 2020, have not paid one or more rental payme...

[3] CA Health & Safety Code § 1569.683 (2021)
    RRF Score: 0.500
    1569.683.  (a) In addition to complying with other applicable regulations, a licensee of a residential care facility for the elderly who sends a notice of eviction to a resident shall set forth in the...

[4] CA Civ Code § 2924n (2021)
    RRF Score: 0.267
    2924n.  Nothing in this article shall relieve a person deemed the legal owner of real property when the trustee’